In [2]:
# ============================================================
# SwissParl API (swissparlpy) -> Business 2025 (DE only)
# BusinessType in (5, 6, 8, 14)
# + robust text cleaning integrated
# ============================================================

# If needed (first time only):
# %pip install -U swissparlpy pandas pyarrow

import pandas as pd
import swissparlpy as spp
import re
import html
import time
from datetime import datetime, timezone

# ----------------------------
# PARAMETERS
# ----------------------------
YEAR = 2025
LANGUAGE = "DE"
BUSINESS_TYPES = (5, 6, 8, 14)
SLEEP_S = 0.2   # polite delay between monthly queries

COLS_WANTED = [
    "ID",
    "SubmissionDate",
    "Language",
    "BusinessShortNumber",
    "BusinessType",
    "BusinessTypeName",
    "Title",
    "SubmittedText",
    "SubmittedBy",
    "FederalCouncilResponseText",
    "ResponsibleDepartmentAbbreviation",
    "TagNames",
]

TEXT_COLS = [
    "Title",
    "SubmittedText",
    "SubmittedBy",
    "FederalCouncilResponseText",
    "TagNames",
]

# ----------------------------
# TEXT CLEANING
# ----------------------------
def clean_text(x: str) -> str:
    """
    Clean SwissParl text fields:
    - HTML / XML unescape
    - remove tags
    - normalize whitespace
    - strip typographic artifacts
    """
    if not isinstance(x, str):
        return ""

    x = html.unescape(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = x.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x)
    x = x.strip().strip('"\'' "“”„’‚")

    return x

# ----------------------------
# DATA FETCH
# ----------------------------
def fetch_businesses_2025(
    year: int = YEAR,
    language: str = LANGUAGE,
    business_types: tuple[int, ...] = BUSINESS_TYPES,
) -> pd.DataFrame:

    rows = []

    for month in range(1, 13):
        start = datetime(year, month, 1, tzinfo=timezone.utc)
        end = (
            datetime(year + 1, 1, 1, tzinfo=timezone.utc)
            if month == 12
            else datetime(year, month + 1, 1, tzinfo=timezone.utc)
        )

        bt_filter = " or ".join([f"BusinessType eq {bt}" for bt in business_types])
        filter_str = f"({bt_filter})"

        data = spp.get_data(
            "Business",
            Language=language,
            SubmissionDate__gte=start,
            SubmissionDate__lt=end,
            filter=filter_str,
        )

        rows.extend(list(data))
        time.sleep(SLEEP_S)

    df = pd.DataFrame(rows)

    # Ensure requested columns exist
    for col in COLS_WANTED:
        if col not in df.columns:
            df[col] = pd.NA

    df = (
        df[COLS_WANTED]
        .drop_duplicates(subset=["ID", "Language"])
        .reset_index(drop=True)
    )

    return df

# ----------------------------
# RUN PIPELINE
# ----------------------------
df_business_2025 = fetch_businesses_2025()

# Clean textual columns
for col in TEXT_COLS:
    if col in df_business_2025.columns:
        df_business_2025[col] = df_business_2025[col].apply(clean_text)

# Display sanity check
pd.set_option("display.max_colwidth", 200)
df_business_2025.head()

# Optional save
# df_business_2025.to_parquet("swissparl_business_2025_DE_clean.parquet", index=False)
# df_business_2025.to_csv("swissparl_business_2025_DE_clean.csv", index=False)


,ID,SubmissionDate,Language,BusinessShortNumber,BusinessType,BusinessTypeName,Title,SubmittedText,SubmittedBy,FederalCouncilResponseText,ResponsibleDepartmentAbbreviation,TagNames
0,20253000,2025-01-09 00:00:00+00:00,DE,25.3000,5,Motion,Kapazitätserweiterung der Nitrochemie,"Der Bundesrat wird beauftragt, zeitgerecht die Voraussetzungen dafür zu schaffen, dass der RUAG MRO das notwendige Kapital für die geplante Kapazitätserweiterung der Nitrochemie bis im Mai 2025 zu...",,Der Bundesrat beantragt die Annahme der Motion.,VBS,Sicherheitspolitik|Wirtschaft
1,20253001,2025-01-10 00:00:00+00:00,DE,25.3001,5,Motion,Eine robuste und resiliente Gesundheitsversorgung in allen Lagen,"Der Bundesrat wird beauftragt, gemeinsam mit den Kantonen eine Strategie zu erarbeiten, wie das Gesundheitswesen im Krisen-, Katastrophen- und Kriegsfall eine robuste und resiliente Versorgung von...",,"Der Bundesrat teilt das Anliegen der Motion, wonach eine umfassende Vorbereitung des Gesundheitswesens auf Ausnahmesituationen wie Krisen, Grossereignisse, Katastrophen oder Krieg von zentraler Wi...",VBS,Sicherheitspolitik|Gesundheit
2,20253002,2025-01-13 00:00:00+00:00,DE,25.3002,6,Postulat,Ex-post-Nachhaltigkeitsanalyse zum Handels- und Wirtschaftspartnerschaftsabkommen zwischen den EFTA-Staaten und Indien,"Der Bundesrat wird beauftragt, beim Handels- und Wirtschaftspartnerschaftsabkommen zwischen den EFTA-Staaten und Indien eine Ex-post Nachhaltigkeitsanalyse vorzunehmen, welche mindestens folgende ...",,"Der Bundesrat hat in seiner Antwort zur Interpellation 24.3260 Molina ausgeführt, warum beim Handels- und Wirtschaftspartnerschaftsabkommen zwischen den EFTA-Staaten und Indien keine ex-ante Nachh...",WBF,Internationale Politik|Wirtschaft
3,20253003,2025-01-14 00:00:00+00:00,DE,25.3003,5,Motion,Auch Navigationssysteme müssen einen Beitrag für die Sicherheit leisten,"Der Bundesrat wird beauftragt die gesetzlichen Grundlagen zu schaffen, damit Betreiber von Navigationsgeräten angeordnete Strassensperrungen abbilden müssen. Eine Minderheit der Kommission (Schill...",,"Je nach Zuständigkeit können die Kantone oder die Gemeinden bei Stau auf der Autobahn temporäre Fahrverbote anordnen. Damit die Anordnung rechtsverbindlich ist, müssen diese temporären Fahrverbote...",UVEK,Medien und Kommunikation|Verkehr
4,20253004,2025-01-14 00:00:00+00:00,DE,25.3004,5,Motion,Schaffung der gesetzlichen Grundlagen zur Verbesserung des Verkehrsmanagements auf den Nord-Süd-Achsen,"Der Bundesrat wird beauftragt, die gesetzlichen und verordnungstechnischen Grundlagen so anzupassen, dass die vom Ausweichverkehr auf den Nord-Süd-Transitachsen betroffenen Kantone bei starker Übe...",,"Die Durchgangsstrassen sollen sicherstellen, dass alle Regionen der Schweiz für den Motorfahrzeugverkehr erreichbar bleiben. Wo die Durchgangsstrassen parallel zu Autobahnen verlaufen, dienen sie ...",UVEK,Verkehr


In [ ]:
import pandas as pd

CSV_PATH = "df_business_2025.csv"
df_questions_2025 = pd.read_csv(CSV_PATH)

df_questions_2025 = df_questions_2025[~df_questions_2025["BusinessTypeName"].isin(["Motion", "Postulat","Interpellation"])]



In [9]:
# ============================================================
# APERTUS LLM CLASSIFICATION (OpenAI-compatible HTTP)
# Dataset: 1 BUSINESS per row (CSV)
# Task: detect (1) sentiment towards public administration (NEG/NEU/POS)
#       (2) populism (YES/NO) people vs elite rhetoric
# Features: robust HTTP retries, parallelism, checkpointing, resume, JSON salvage
# ============================================================

import os
import json
import re
import time
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# -------------------------------
# 0) INPUT / OUTPUT
# -------------------------------
CSV_PATH = "df_business_2025.csv"

OUTPUT_PARQUET = Path("df_business_2025_labeled.parquet")
OUTPUT_CSV     = Path("df_business_2025_labeled.csv")  # optional export at end

CHECKPOINT_EVERY = 25
PRINT_EVERY = 25


# -------------------------------
# 1) APERTUS SERVER CONFIG
# -------------------------------
APERTUS_BASE_URL = os.getenv("APERTUS_BASE_URL", "http://127.0.0.1:8080").rstrip("/")
CHAT_URL   = f"{APERTUS_BASE_URL}/v1/chat/completions"
MODELS_URL = f"{APERTUS_BASE_URL}/v1/models"

REQ_TIMEOUT = int(os.getenv("REQ_TIMEOUT", "120"))

SESSION = requests.Session()

# ✅ Robust HTTP retries (overnight stability)
retry = Retry(
    total=int(os.getenv("HTTP_TOTAL_RETRIES", "5")),
    backoff_factor=float(os.getenv("HTTP_BACKOFF_FACTOR", "0.5")),
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
    raise_on_status=False,
)
adapter = HTTPAdapter(
    max_retries=retry,
    pool_connections=int(os.getenv("HTTP_POOL_CONNECTIONS", "50")),
    pool_maxsize=int(os.getenv("HTTP_POOL_MAXSIZE", "50")),
)
SESSION.mount("http://", adapter)
SESSION.mount("https://", adapter)


def get_server_model_id(default: str = "local-model") -> str:
    """Fetch model id from /v1/models. Falls back to env APERTUS_MODEL, then default."""
    env_model = os.getenv("APERTUS_MODEL")
    if env_model:
        return env_model

    try:
        r = SESSION.get(MODELS_URL, timeout=REQ_TIMEOUT)
        r.raise_for_status()
        data = r.json()
        models = data.get("data", [])
        if isinstance(models, list) and models:
            mid = models[0].get("id")
            if mid:
                return mid
    except Exception:
        pass

    return default


MODEL_ID = get_server_model_id()
print(f"✅ APERTUS base_url={APERTUS_BASE_URL} | model='{MODEL_ID}'")


# -------------------------------
# 2) GENERATION CONFIG
# -------------------------------
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))
TOP_P       = float(os.getenv("TOP_P", "1.0"))
MAX_TOKENS  = int(os.getenv("MAX_TOKENS", "180"))

MAX_RETRIES = int(os.getenv("MAX_RETRIES", "2"))

# Speed cap
MAX_CHARS_INPUT = int(os.getenv("MAX_CHARS_INPUT", "2000"))

# Concurrency
N_WORKERS = int(os.getenv("N_WORKERS", "2"))
MAX_IN_FLIGHT_MULT = int(os.getenv("MAX_IN_FLIGHT_MULT", "5"))  # max pending futures = N_WORKERS * this


# -------------------------------
# 3) LABEL SPACE
# -------------------------------
SENTIMENTS = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
POPULISM   = ["YES", "NO"]


# -------------------------------
# 4) PROMPT (you can tweak content freely)
# -------------------------------
SYSTEM_PROMPT = (
    "You are a strict political communication classifier.\n"
    "Return ONLY a valid JSON object and nothing else.\n"
)

def build_user_prompt(title: str, submitted_text: str, language: str) -> str:
    # Keep it small and focused on the "business" statement
    return f"""
You will analyze ONE statement related to Swiss public authorities.

TASKS

1) Sentiment toward PUBLIC ADMINISTRATION
Choose ONE label: {SENTIMENTS}
Public administration includes: federal/cantonal/communal administrations, offices/agencies, civil servants, bureaucracy,
administrative procedures, regulators, public authorities.
Rules:
- NEGATIVE if the statement criticizes, blames, pressures, accuses, or complains about administration/bureaucracy/authorities.
- POSITIVE if it praises/supports administrative action.
- NEUTRAL if it is descriptive or targets politicians/parties in general without evaluating administrative functioning.

2) Populism detection
Choose ONE label: {POPULISM}
YES only if it frames politics as "the pure people" vs "the corrupt elite/establishment" (anti-pluralist),
delegitimizes institutions as illegitimate/corrupt, or claims exclusive representation of "the people".
Otherwise NO.

OUTPUT FORMAT (STRICT)
Return ONLY this JSON schema (no extra keys, no markdown):
{{
  "admin_sentiment": "<NEGATIVE|NEUTRAL|POSITIVE>",
  "populism": "<YES|NO>"
}}

STATEMENT
Language: {language}
Title: {title}
SubmittedText: {submitted_text}
""".strip()


# -------------------------------
# 5) HELPERS (clamp, JSON salvage, validation)
# -------------------------------
def clamp_text(s: str, max_chars: int) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) <= max_chars:
        return s
    head = s[: int(max_chars * 0.75)]
    tail = s[-int(max_chars * 0.25):]
    return head + " ... " + tail

_JSON_BLOB_RE = re.compile(r"\{.*\}", re.DOTALL)

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None
    text = text.strip()

    # Best case: single-line JSON
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{") and line.endswith("}"):
            try:
                obj = json.loads(line)
                return obj if isinstance(obj, dict) else None
            except Exception:
                pass

    # Fallback: first { ... } blob
    m = _JSON_BLOB_RE.search(text)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

DEFAULT_RES = {"admin_sentiment": "NEUTRAL", "populism": "NO"}

def normalize_and_validate(obj: Dict[str, Any]) -> Dict[str, Any]:
    sent = str(obj.get("admin_sentiment", "")).strip().upper()
    pop  = str(obj.get("populism", "")).strip().upper()

    if sent not in SENTIMENTS:
        sent = DEFAULT_RES["admin_sentiment"]
    if pop not in POPULISM:
        pop = DEFAULT_RES["populism"]

    return {"admin_sentiment": sent, "populism": pop}


# -------------------------------
# 6) SERVER CALLS
# -------------------------------
def chat_completion(system_prompt: str, user_prompt: str) -> str:
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_TOKENS,
    }

    # If supported by your server, uncomment to enforce JSON:
    # payload["response_format"] = {"type": "json_object"}

    r = SESSION.post(CHAT_URL, json=payload, timeout=REQ_TIMEOUT)
    r.raise_for_status()
    j = r.json()
    return j["choices"][0]["message"]["content"]

def llm_classify_one(title: str, submitted_text: str, language: str) -> Dict[str, Any]:
    submitted_text = clamp_text(submitted_text or "", MAX_CHARS_INPUT)
    user_prompt = build_user_prompt(title or "", submitted_text, language or "")

    for attempt in range(MAX_RETRIES + 1):
        try:
            text = chat_completion(SYSTEM_PROMPT, user_prompt)
            obj = extract_json_object(text)
            if obj is not None:
                return normalize_and_validate(obj)
        except Exception:
            time.sleep(0.4 * (attempt + 1))

    return dict(DEFAULT_RES)


# -------------------------------
# 7) CHECKPOINTING
# -------------------------------
LABEL_COLS = ["admin_sentiment_llm", "populism_llm"]

def atomic_save_parquet(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


# -------------------------------
# 8) APPLY TO df_business (parallel + resume)
# -------------------------------
def classify_dataframe(df_in: pd.DataFrame, limit: Optional[int] = None) -> pd.DataFrame:
    # ✅ Resume if checkpoint exists
    if OUTPUT_PARQUET.exists():
        df = pd.read_parquet(OUTPUT_PARQUET)
        print(f"✅ Resuming from checkpoint: {OUTPUT_PARQUET}  shape={df.shape}")
    else:
        df = df_in.copy()

    if limit is not None:
        df = df.head(limit).copy()

    # Ensure label columns exist
    for c in LABEL_COLS:
        if c not in df.columns:
            df[c] = pd.NA

    # Choose rows pending classification
    pending_indices = [i for i in df.index if pd.isna(df.at[i, "admin_sentiment_llm"])]
    total_pending = len(pending_indices)
    total_all = len(df)
    already_done = total_all - total_pending
    print(f"📌 Pending: {total_pending} | Already done: {already_done} | Total: {total_all}")

    if total_pending == 0:
        return df

    # Row -> payload
    def payload_for_idx(idx) -> Tuple[str, str, str]:
        row = df.loc[idx]
        title = str(row.get("Title", "") or "")
        submitted = str(row.get("SubmittedText", "") or "")
        lang = str(row.get("Language", "") or "")
        return title, submitted, lang

    max_in_flight = max(1, N_WORKERS * MAX_IN_FLIGHT_MULT)
    done_since_save = 0
    done_total = already_done

    def checkpoint_save():
        atomic_save_parquet(df, OUTPUT_PARQUET)
        print(f"💾 checkpoint saved -> {OUTPUT_PARQUET}")

    try:
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            it = iter(pending_indices)
            future_to_idx: Dict[Any, Any] = {}

            # Prime queue
            for _ in range(min(max_in_flight, total_pending)):
                idx = next(it, None)
                if idx is None:
                    break
                t, s, l = payload_for_idx(idx)
                future_to_idx[ex.submit(llm_classify_one, t, s, l)] = idx

            while future_to_idx:
                done_set, _ = wait(future_to_idx.keys(), return_when=FIRST_COMPLETED)

                for fut in done_set:
                    idx = future_to_idx.pop(fut)

                    try:
                        res = fut.result()
                    except Exception:
                        res = dict(DEFAULT_RES)

                    df.at[idx, "admin_sentiment_llm"] = res["admin_sentiment"]
                    df.at[idx, "populism_llm"] = res["populism"]

                    done_since_save += 1
                    done_total += 1

                    if done_total % PRINT_EVERY == 0 or done_total == total_all:
                        print(f"… classified {done_total}/{total_all} (session: {done_since_save}/{total_pending})")

                    if done_since_save % CHECKPOINT_EVERY == 0:
                        checkpoint_save()

                    # refill one slot
                    nxt = next(it, None)
                    if nxt is not None:
                        t, s, l = payload_for_idx(nxt)
                        future_to_idx[ex.submit(llm_classify_one, t, s, l)] = nxt

    except KeyboardInterrupt:
        print("\n🛑 Interrupted by user. Saving checkpoint before exit…")
        checkpoint_save()
        raise
    except Exception as e:
        print(f"\n🛑 Fatal error: {e}. Saving checkpoint…")
        checkpoint_save()
        raise

    checkpoint_save()
    return df


# -------------------------------
# 9) RUN
# -------------------------------
df_business = pd.read_csv(CSV_PATH)

# Optional: filter rows with no text
if "SubmittedText" in df_business.columns:
    df_business = df_business[df_business["SubmittedText"].notna()].copy()

# 🔧 TEST RUN:
# df_labeled = classify_dataframe(df_business, limit=25)

# ✅ FULL RUN:
df_labeled = classify_dataframe(df_business, limit=100)

print("✅ df_labeled:", df_labeled.shape)
print(df_labeled[["Title", "Language", "admin_sentiment_llm", "populism_llm"]].head(10))

# Optional export:
# df_labeled.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
# print(f"✅ Saved CSV: {OUTPUT_CSV}")


✅ APERTUS base_url=http://127.0.0.1:8080 | model='swiss-ai_Apertus-8B-Instruct-2509-Q6_K.gguf'
📌 Pending: 100 | Already done: 0 | Total: 100
… classified 25/100 (session: 25/100)
💾 checkpoint saved -> df_business_2025_labeled.parquet
… classified 50/100 (session: 50/100)
💾 checkpoint saved -> df_business_2025_labeled.parquet
… classified 75/100 (session: 75/100)
💾 checkpoint saved -> df_business_2025_labeled.parquet
… classified 100/100 (session: 100/100)
💾 checkpoint saved -> df_business_2025_labeled.parquet
💾 checkpoint saved -> df_business_2025_labeled.parquet
✅ df_labeled: (100, 13)
                                                                                                                                                              Title  \
0                                                                                             Europäischer Gerichtshof und die Schweiz: Urteile des Schiedsgerichts   
1                                                       Abkommen zwische